# Facility factories & execution control

`coffea-workflow` factory abstraction: what it solves, what it exposes to users, and where are the limits of what can be controlled.

---
## 1. The problem: analysis code shouldn't know where it runs

A typical HEP analysis demands writing the processor once, but the execution scripts for each facility can be rewritten multiple times depending on facility:

```
local execution - FuturesExecutor, 4 workers
coffea-casa - Client("tls://localhost:8786"), DaskExecutor
lxplus - HTCondorCluster(...), scale(10), DaskExecutor
```

Each of these files would be mostly a boilerplate: Dask client setup, proxy checks, HTCondor job parameters.

**Goal of `coffea-workflow` factories: isolate all execution infrastructure into a single config object, so analysis code is written once and runs anywhere.**

In [ ]:
from coffea_workflow import Step, Workflow, Fileset, Analysis, Plotting, RunConfig, ExecutorConfig, run
from coffea_workflow import facilities
from coffea_workflow.facilities import CoffeaCasaFactory, LxplusFactory
from analysis import get_fileset, run_analysis, plot_results

---
## 2. Workflow that can be reused on different facilities

#### Reminder about the abstraction:
Fileset, Analysis, and Plotting are the only opinionated steps that expect certain output/input. CustomArtifact type supports adding unopinionated steps that only expect the user function to be provided and tracks the output/input dependency.

TODO: communicate the output and input better through the ux.

Moreover, `Analysis` is the step that provides more control over the execution under the hood through RunConfig (described later). `Fileset` step that doesn't require the input. `Plotting` always rerun to produce the image.


In [ ]:
step_fileset  = Step(name="Fileset",  step_type=Fileset,   builder=get_fileset)
step_analysis = Step(name="Analysis", step_type=Analysis,  builder=run_analysis) # runner config, extract runner to workflow level
step_plotting = Step(name="Plotting", step_type=Plotting,  builder=plot_results)

workflow = Workflow()
workflow.add(step_fileset)
workflow.add(step_analysis,  depends_on=[step_fileset])
workflow.add(step_plotting,  depends_on=[step_analysis])

---
## 3. RunConfig: how + where

The entire execution infrastructure lives in `RunConfig`. Swapping facilities means changing one argument. The workflow and analysis code are not touched.

Step-by-step:
1. Check the facility; set it up
2. Initiate the executor from executor_config for a given facility
3. Split the fileset if split strategies were mentioned
4. Sequentially execute each subset one after another, summing the results and caching successful results while marking the subsets to be rerun.

In [ ]:
# Local development 
# no Dask, no cluster, no setup; good for debugging and small tests

config = RunConfig(
    strategy="by_dataset",
    percentage=20, # 5 subsets per one dataset
    cache_dir=".cache_local",
    facility=facilities.local,
    executor_config=ExecutorConfig(executor_type="FuturesExecutor", workers=4),
)

run(workflow, config)

In [ ]:
# CoffeaCasa
# DaskExecutor: connects to the pre-configured scheduler at tls://localhost:8786.
# CoffeaCasaFactory handles the Client setup

config = RunConfig(
    strategy="by_dataset",
    percentage=20,
    cache_dir=".cache_coffea_casa",
    facility=CoffeaCasaFactory(),
    executor_config=ExecutorConfig(executor_type="DaskExecutor"),
)
run(workflow, config)

In [ ]:
# lxplus (CERN HTCondor)
# DaskExecutor
# The factory handles: proxy check, HTCondorCluster setup, scale(), client creation.

config = RunConfig(
    strategy="by_dataset",
    cache_dir=".cache_lxplus",
    facility=LxplusFactory(
        worker_image="~/worker.sif",   # Apptainer image with coffea pre-installed
        queue="longlunch",             # HTCondor queue: espresso=20m, longlunch=2h, workday=8h
        workers=10,                   # number of HTCondor jobs to submit
    ),
    executor_config=ExecutorConfig(executor_type="DaskExecutor"),
)
run(workflow, config)

---
## 4. What a factory actually does

Every factory implements three methods:

`preflight()`: check all prerequisites before any compute starts. Raises `RuntimeError` with the exact fix command if something is missing (expired VOMS proxy, missing image, wrong hostname, ...). 

`build(ec)`: create and return the coffea executor (`DaskExecutor`, `FuturesExecutor`, ...). Handles Client construction, HTCondorCluster setup, worker scaling, package installation.

`close()`: tear down resources: cancel HTCondor jobs, disconnect Dask client. Called automatically at the end of `run()`. 

**`LxplusFactory` additional parameters:**
- `worker_image` — Apptainer `.sif` path; the entire Python environment (coffea, analysis code, dependencies) is baked in at build time — no runtime installation needed
- `queue` — HTCondor job flavour: `espresso` (20 min), `longlunch` (2h), `workday` (8h)
- `workers` — number of HTCondor slots to request
- `extra_pythonpath` — override packages inside the container at runtime (for development, without rebuilding the image)

**Building the worker image** with the help of additional utility:


In [ ]:
from coffea_workflow.facilities import generate_apptainer_def
generate_apptainer_def(output="worker.def", extra_packages=("correctionlib==2.3.0",))
# prints the instruction on how to build apptainer

## 5. Fileset splitting: control over subset + partial result preservation

## 6. How DaskExecutor distributes work within a chunk

By default it's sequential process right now. So one subset is broken into coffea chunks and all workers share the load simultanouesly.
**What the user controls at this level:**
- `chunksize` — set inside the analysis function when calling `Runner(chunksize=...)` (default 100k events). Larger = fewer tasks, less overhead, more memory per worker. Smaller = more tasks, better load balancing.
- `workers` — via `LxplusFactory(workers=N)` or the coffea-casa cluster size. More workers = more parallelism within a chunk.

**Question: if the user wants one worker to treat more than one file as one job, is it possible to set uit p by just increasing the coffea chunk-size**

## 7. Sequential vs parallel chunk submission
By default, `coffea-workflow` processes chunks **sequentially**: all workers collaborate on subset 1, then subset 2, etc. (done to preserve partial result even if something breaks)

Alternative: `clien.submit` to process in parallel

`N > M` (more workers than chunks) - **Sequential** - all workers collaborate per chunk; parallel leaves N-M workers without work

`N ≈ M` - Equal - parallel has slightly less submission overhead 

`N < M` - **Parallel** - parallel keeps workers busy; sequential has idle gaps between rounds 

Chunks have **unequal** file counts - **Sequential** - otherwise one subsest can be processed much quicker and the workers are just chilling

Chunks are equal in size - Parallel is fine, no imbalance

**Question: whether adding a parameter of `parallel` or `sequential` submission to `RunConfig` is meaningful**